# Data Lake

In [ ]:
import os

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv

load_dotenv('../../../.env_file/infra.env')

True

## Generate data

In [3]:
data_dict = {
    'name': [''.join(np.random.choice(list('abcdefghijklmnopqrstuvwxyz'), size=8)) for _ in range(100)],
    'price': np.random.random(100).tolist(),
    'quantity': np.random.randint(1, 10_000, size=100).tolist(),
}

df = pd.DataFrame(data_dict)
df.head()

,name,price,quantity
0,liwqfvsy,0.252691,7424
1,uavllcfm,0.004512,4326
2,dewpsros,0.199728,3694
3,skooxenu,0.174736,4752
4,ocrjnvoh,0.228955,685


In [4]:
split_idx = np.array_split(df.index, 3)
df_part_1, df_part_2, df_part_3 = [df.loc[idx].reset_index(drop=True) for idx in split_idx]
df_part_1, df_part_2, df_part_3

(        name     price  quantity
 0   liwqfvsy  0.252691      7424
 1   uavllcfm  0.004512      4326
 2   dewpsros  0.199728      3694
 3   skooxenu  0.174736      4752
 4   ocrjnvoh  0.228955       685
 5   zviebsbm  0.123101      5897
 6   votlkfjl  0.356877      6113
 7   zmeimwcr  0.543656      8083
 8   mvzfshgz  0.616986      9557
 9   gyejiysw  0.946289       313
 10  rcclequo  0.730737      9758
 11  gyjcynfi  0.546401      1795
 12  zecbosgi  0.360825       382
 13  lhxsltmn  0.498690      6205
 14  eruhgfxo  0.739164      4185
 15  ysucmgmx  0.601977      5492
 16  zvcigdur  0.109989      8229
 17  yuywwbio  0.030219      3412
 18  walkzyma  0.751705      5310
 19  keyusbgw  0.787373      5718
 20  vjgsneyn  0.725399      3819
 21  ykqjxlzw  0.131417       843
 22  vvyfhmjn  0.906111      7065
 23  fargcgov  0.113002      5599
 24  yikjjhei  0.683471      7179
 25  xvdqxrjk  0.985338      9748
 26  ixgixbsh  0.280599      4583
 27  wcuodkgn  0.426714      4184
 28  sbydpcnv 

In [ ]:
all_df:list[pd.DataFrame] = [df_part_1, df_part_2, df_part_3]
for idx, part in enumerate(all_df):
    part.to_parquet(f'../../../data/test/dl/part_{idx}.parquet')

## Push data to S3

In [7]:
def create_bucket(bucket_name, region='ap-northeast-1'):
    """Create an S3 bucket in a specified region

    If a region is not specified, the bucket is created in the S3 default
    region (us-east-1).

    :param bucket_name: Bucket to create
    :param region: String region to create bucket in, e.g., 'us-west-2'
    :return: True if bucket created, else False
    """

    # Create bucket
    try:
        bucket_config = {}
        s3_client = boto3.client(
            's3',
            region_name=region,
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        if region != 'us-east-1':
            bucket_config['CreateBucketConfiguration'] = {'LocationConstraint': region}

        s3_client.create_bucket(Bucket=bucket_name, **bucket_config)
    except ClientError as e:
        print(e)
        return False
    return True

def upload_file(file_name, bucket, object_name=None):
    """Upload a file to an S3 bucket

    :param file_name: File to upload
    :param bucket: Bucket to upload to
    :param object_name: S3 object name. If not specified then file_name is used
    :return: True if file was uploaded, else False
    """

    # If S3 object_name was not specified, use file_name
    if object_name is None:
        object_name = os.path.basename(file_name)

    s3_client = boto3.client(
        's3',
        region_name='ap-northeast-1',
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
    )
    try:
        response = s3_client.upload_file(file_name, bucket, object_name)
    except ClientError as e:
        print(e)
        return False
    return True

# create_bucket('flezi-agent', 'ap-northeast-1')
# upload_file('../../data/test/csv/04.customer.csv', 'flezi-test', 'nyc-taxi/04.customer.csv')

In [19]:
s3_client = boto3.client(
    's3',
    region_name='ap-northeast-1',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

bucket_name = 'flezi-data'
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Found bucker {bucket_name}. Skip creation')
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == '404':
        print(f'No bucket {bucket_name} found. Create a new one')
        create_bucket(bucket_name)

Found bucker flezi-data. Skip creation


In [ ]:
for fn in os.listdir('../../../data/test/dl'):
    fp = os.path.join('../../../data/test/dl', fn)
    print(f"Uploading {fp}")
    upload_file(fp, bucket_name, os.path.join('product', fn))

Uploading ../../data/test/dl/part_0.parquet
Uploading ../../data/test/dl/part_1.parquet
Uploading ../../data/test/dl/part_2.parquet


## Query from Trino

- Access to `Trino`:

    ```bash
    docker exec -it datalake-trino /bin/bash
    trino
    ```

- Run the following SQL queries:

    ```sql
    CREATE SCHEMA IF NOT EXISTS datalake.store
        WITH (location = 's3://nblong-data-project/nyc_taxi//');

    CREATE TABLE IF NOT EXISTS datalake.store.product ( -- catalog.schema.table
        "name" VARCHAR(20),
        "price" DOUBLE,
        "quantity" INT
    ) WITH (
        external_location = 's3://nblong-data-project/nyc_taxi//product',
        format = 'PARQUET'
    );
    ```